<a href="https://colab.research.google.com/github/y123ash/KNOVOS_Sentiment_Analysis/blob/main/Sentiment_Analysis_Sep16_2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
i am currentlu work to trying sentiment analysis. i have tried to do self knwledge distil. (betrt to bert), fine tunning nbert, can you tell me the approx acuuracy aof classififcation of sentiment. rewpectively. and what else can i do, to imporove accutrcacy of sresultant sentime nt model (how to make that mode) fd
==========================================

For sentiment analysis using BERT-based models, the expected classification accuracy varies depending on the dataset, fine-tuning quality, and additional techniques like knowledge distillation. Here's a general idea:

Approximate Accuracy Ranges:
Fine-Tuning BERT (Without Knowledge Distillation):

Expected Accuracy: 85% - 90% on large, well-structured datasets like IMDb or SST-2.
Challenges: Without additional techniques, fine-tuning may overfit or underfit on smaller datasets.
Self-Knowledge Distillation (BERT to BERT):

Expected Accuracy: Slightly lower (around 80% - 85%) if you haven't fully optimized it, because distilling knowledge from the same model (self-distillation) can sometimes result in marginal gains or even a slight loss in performance if not done properly.
Challenges: If not tuned well, distillation might not improve or could even hurt performance, especially without sufficient training data or a proper loss function.
What Else Can You Do to Improve Accuracy:
To improve the accuracy of your sentiment analysis model, you can explore the following strategies:

Knowledge Distillation with Different Architectures:

Why: Instead of using BERT-to-BERT, try distilling knowledge from a larger, more powerful model (like RoBERTa or GPT) into a smaller BERT variant (like DistilBERT).
How: Use a more accurate teacher model (e.g., roberta-large) to guide a smaller BERT model. This can improve accuracy while reducing inference time.
Ensemble Models:

Why: Combining multiple models (like BERT and RoBERTa) can smooth out individual model biases and give more robust predictions.
How: Train several models (BERT, RoBERTa, XLNet) and average their predictions or use majority voting to improve overall accuracy.
Data Augmentation:

Why: If your dataset is small or unbalanced, using techniques like back-translation or synonym replacement can create more diverse training samples, improving generalization.
How: Augment your dataset by creating paraphrases of sentences and fine-tuning your model on this larger, more diverse dataset.
Using Pre-trained Language Models with Better Performance:

Why: Some models, like RoBERTa and DeBERTa, tend to perform better than BERT on text classification tasks.
How: Fine-tune models like roberta-large or deberta-large that have demonstrated superior performance on benchmarks like GLUE for sentiment analysis.
Hyperparameter Tuning (Learning Rate, Batch Size, etc.):

Why: Optimizing hyperparameters like learning rate, batch size, and the number of epochs can lead to significant performance improvements.
How: Use grid search or tools like Optuna to automatically tune your model’s hyperparameters.
Learning Rate Schedulers:

Why: A constant learning rate may not be optimal throughout training. A scheduler can help you gradually reduce the learning rate to improve convergence.
How: Use a scheduler like linear warmup or cosine decay during training to adjust the learning rate dynamically.
Regularization Techniques:

Why: Adding dropout or weight decay can help prevent overfitting, especially when fine-tuning on small datasets.
How: Increase the dropout rate in the model or use weight decay in the optimizer to help regularize the training.
Balanced Dataset and Weighted Loss:

Why: Imbalanced data can skew results. Adjusting the loss function by applying weights to each class can improve the performance on under-represented classes.
How: Calculate class weights and use them in your loss function (e.g., CrossEntropyLoss with class weights).
Cross-Lingual Sentiment Analysis:

Why: If you're dealing with multiple languages, use multilingual BERT models (XLM-Roberta or mBERT), which can handle sentiment analysis across different languages and may offer better performance for multilingual datasets.
Advanced Pre-Training Techniques:

Why: Fine-tuning after more advanced pre-training (like domain-specific pre-training on sentiment-rich corpora) can significantly boost performance.
How: Further pre-train BERT on sentiment-specific datasets (e.g., movie reviews or product reviews) before fine-tuning it on your target dataset.
Example for Applying These Strategies:
Here’s how you can combine some of these techniques to improve your sentiment model:

python
Copy code
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

# Load the tokenizer and pre-trained model
tokenizer = AutoTokenizer.from_pretrained('roberta-base')
model = AutoModelForSequenceClassification.from_pretrained('roberta-base', num_labels=3)

# Data augmentation (if necessary, use synonym replacement or back-translation)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_total_limit=1,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
)

# Define a Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,  # Your tokenized training dataset
    eval_dataset=eval_dataset,    # Your tokenized evaluation dataset
    compute_metrics=compute_metrics,
)

# Fine-tune the model
trainer.train()

# Optional: Apply distillation, ensemble, or weighted loss techniques after fine-tuning.
Next Steps:
Start by fine-tuning RoBERTa or DeBERTa for better baseline performance.
Try knowledge distillation with a larger model (like roberta-large) as the teacher.
Experiment with data augmentation and hyperparameter tuning.
With these approaches, you can expect improved accuracy for your sentiment classification task. Let me know if you want further details on implementing any of these strategies!


In [ ]:
# Install necessary libraries
#!pip install transformers datasets torch

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("glue", "mnli")

# Load tokenizers
teacher_tokenizer = AutoTokenizer.from_pretrained("roberta-large")
student_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenize the dataset
def preprocess_function(examples):
    return teacher_tokenizer(examples['premise'], examples['hypothesis'], truncation=True, padding=True, max_length=128)

# Tokenizing the dataset
tokenized_datasets = dataset.map(preprocess_function, batched=True)

# Prepare the models
teacher_model = AutoModelForSequenceClassification.from_pretrained("roberta-large", num_labels=3)
student_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3)

# Freeze teacher model's parameters
for param in teacher_model.parameters():
    param.requires_grad = False

# Custom Trainer with knowledge distillation
class DistillationTrainer(Trainer):
    def __init__(self, teacher_model, *args, temperature=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher = teacher_model
        self.temperature = temperature
        self.teacher.eval()

    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        student_outputs = model(**inputs)
        student_logits = student_outputs.logits

        with torch.no_grad():
            teacher_outputs = self.teacher(**inputs)
            teacher_logits = teacher_outputs.logits

        # Distillation loss using KL Divergence
        loss = F.kl_div(
            F.log_softmax(student_logits / self.temperature, dim=-1),
            F.softmax(teacher_logits / self.temperature, dim=-1),
            reduction="batchmean"
        ) * (self.temperature ** 2)

        # Cross-entropy loss
        loss += F.cross_entropy(student_logits, labels)

        return (loss, student_outputs) if return_outputs else loss

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
)

# Initialize the distillation trainer
trainer = DistillationTrainer(
    teacher_model=teacher_model,
    model=student_model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation_matched'],
    tokenizer=student_tokenizer,
    temperature=2.0  # Higher temperature smooths the probabilities from the teacher
)

# Train the student model
trainer.train()

# Evaluate the student model on the validation set
trainer.evaluate()

# Example for prediction
def predict_nli(text_premise, text_hypothesis):
    inputs = student_tokenizer(text_premise, text_hypothesis, return_tensors="pt", truncation=True, padding=True)
    outputs = student_model(**inputs)
    predicted_class = torch.argmax(outputs.logits, dim=-1).item()

    # Label map according to MNLI
    label_map = {-1: 'entailment', 1: 'neutral', 2: 'contradiction'}
    return label_map[predicted_class]

# Example usage
text_premise = "The dog is flying in the park."
text_hypothesis = "The dog is beyond death."
predicted_label = predict_nli(text_premise, text_hypothesis)
print(f"Predicted label: {predicted_label}")

RuntimeError: Failed to import transformers.trainer because of the following error (look up to see its traceback):
Failed to import transformers.integrations.integration_utils because of the following error (look up to see its traceback):
Failed to import transformers.modeling_tf_utils because of the following error (look up to see its traceback):
partially initialized module 'tf_keras.src' has no attribute 'utils' (most likely due to a circular import)

In [ ]:
!pip uninstall tensorflow keras -y

In [ ]:
# Install necessary libraries
#!pip install transformers datasets torch

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("glue", "mnli")

# Load tokenizers
teacher_tokenizer = AutoTokenizer.from_pretrained("roberta-large")
student_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenize the dataset
def preprocess_function(examples):
    return teacher_tokenizer(examples['premise'], examples['hypothesis'], truncation=True, padding=True, max_length=128)

# Tokenizing the dataset
tokenized_datasets = dataset.map(preprocess_function, batched=True)

# Prepare the models
teacher_model = AutoModelForSequenceClassification.from_pretrained("roberta-large", num_labels=3)
student_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3)

# Freeze teacher model's parameters
for param in teacher_model.parameters():
    param.requires_grad = False

# Custom Trainer with knowledge distillation
class DistillationTrainer(Trainer):
    def __init__(self, teacher_model, *args, temperature=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher = teacher_model
        self.temperature = temperature
        self.teacher.eval()

    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")
        student_outputs = model(**inputs)
        student_logits = student_outputs.logits

        with torch.no_grad():
            teacher_outputs = self.teacher(**inputs)
            teacher_logits = teacher_outputs.logits

        # Distillation loss using KL Divergence
        loss = F.kl_div(
            F.log_softmax(student_logits / self.temperature, dim=-1),
            F.softmax(teacher_logits / self.temperature, dim=-1),
            reduction="batchmean"
        ) * (self.temperature ** 2)

        # Cross-entropy loss
        loss += F.cross_entropy(student_logits, labels)

        return (loss, student_outputs) if return_outputs else loss

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
)

# Initialize the distillation trainer
trainer = DistillationTrainer(
    teacher_model=teacher_model,
    model=student_model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation_matched'],
    tokenizer=student_tokenizer,
    temperature=2.0  # Higher temperature smooths the probabilities from the teacher
)

# Train the student model
trainer.train()

# Evaluate the student model on the validation set
trainer.evaluate()

# Example for prediction
def predict_nli(text_premise, text_hypothesis):
    inputs = student_tokenizer(text_premise, text_hypothesis, return_tensors="pt", truncation=True, padding=True)
    outputs = student_model(**inputs)
    predicted_class = torch.argmax(outputs.logits, dim=-1).item()

    # Label map according to MNLI
    label_map = {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
    return label_map[predicted_class]

# Example usage
text_premise = "The dog is flying in the park."
text_hypothesis = "The dog is beyond death."
predicted_label = predict_nli(text_premise, text_hypothesis)
print(f"Predicted label: {predicted_label}")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/52.2M [00:00<?, ?B/s]

(…)alidation_matched-00000-of-00001.parquet:   0%|          | 0.00/1.21M [00:00<?, ?B/s]

(…)dation_mismatched-00000-of-00001.parquet:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


IndexError: index out of range in self

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("glue", "mnli")

# Load tokenizers
teacher_tokenizer = AutoTokenizer.from_pretrained("roberta-large")
student_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Ensure that the labels are mapped correctly: 0 = entailment, 1 = neutral, 2 = contradiction
def preprocess_function(examples):
    inputs = teacher_tokenizer(examples['premise'], examples['hypothesis'], truncation=True, padding=True, max_length=128)
    inputs['labels'] = examples['label']  # No need to remap, as labels are already 0, 1, 2 in the MNLI dataset
    return inputs

# Tokenizing the dataset
tokenized_datasets = dataset.map(preprocess_function, batched=True)

# Prepare the models
teacher_model = AutoModelForSequenceClassification.from_pretrained("roberta-large", num_labels=3)
student_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3)

# Freeze teacher model's parameters
for param in teacher_model.parameters():
    param.requires_grad = False

# Custom Trainer with knowledge distillation
class DistillationTrainer(Trainer):
    def __init__(self, teacher_model, *args, temperature=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher = teacher_model
        self.temperature = temperature
        self.teacher.eval()

    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.pop("labels")  # Extract labels
        student_outputs = model(**inputs)
        student_logits = student_outputs.logits

        with torch.no_grad():
            teacher_outputs = self.teacher(**inputs)
            teacher_logits = teacher_outputs.logits

        # Distillation loss using KL Divergence
        loss = F.kl_div(
            F.log_softmax(student_logits / self.temperature, dim=-1),
            F.softmax(teacher_logits / self.temperature, dim=-1),
            reduction="batchmean"
        ) * (self.temperature ** 2)

        # Cross-entropy loss
        loss += F.cross_entropy(student_logits, labels)

        return (loss, student_outputs) if return_outputs else loss

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
)

# Initialize the distillation trainer
trainer = DistillationTrainer(
    teacher_model=teacher_model,
    model=student_model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation_matched'],
    tokenizer=student_tokenizer,
    temperature=2.0  # Higher temperature smooths the probabilities from the teacher
)

# Train the student model
trainer.train()

# Evaluate the student model on the validation set
trainer.evaluate()

# Example for prediction
def predict_nli(text_premise, text_hypothesis):
    inputs = student_tokenizer(text_premise, text_hypothesis, return_tensors="pt", truncation=True, padding=True)
    outputs = student_model(**inputs)
    predicted_class = torch.argmax(outputs.logits, dim=-1).item()

    # Label map according to MNLI
    label_map = {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
    return label_map[predicted_class]

# Example usage
text_premise = "The dog is flying in the park."
text_hypothesis = "The dog is beyond death."
predicted_label = predict_nli(text_premise, text_hypothesis)
print(f"Predicted label: {predicted_label}")


Map:   0%|          | 0/392702 [00:00<?, ? examples/s]

In [ ]:
pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.3/474.3 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 9.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Uninstalling pyarrow-14.0.2:
      Successfully uninstalled pyarrow-14.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 requires pyarrow<15.0.0a0,>=14.0.1, but you have pyarrow 17.0.0 which is incompatible.
ibis-framework 8.0.0 requires pyarrow<16,>=2, but you have pyarrow 17.0.0 which is incompatible.


In [ ]:
-numpy.lib.stride_tricks.as_strided
-centring an image or object
-imagef using nparrays and
-cannto import image usingg numpu mat;plotlib.pyiplot
-array=np.array([2,3],[4,6],[2,4],[3,10])
-n_ro2, n_col= aray.shape
-array=array.T
-shape=array.shape[:-1]+(array.shape[1]-depth+1)
-depth=3
-strides=array.strides + (array.strides([]))
-np.reshape(np.swapaex(array,0,1), (n-rows-deph+1 + (depth-1)*n_cols)

-binary modues
-bert for sequcen cclassication, weights not initilazlied frompretrained checkpoint: bert-base08-uncased
            -p=np.random.randint(M, size=N)#M=30,
            -X=np.random.rand(N)
            -result_einsum=np.einsum('ijk', k->ji, k[p], X)

SyntaxError: invalid syntax (<ipython-input-1-f0ae21fff33b>, line 2)

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoConfig, Trainer, TrainingArguments)

# Load the data
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
dataset2 = pd.read_parquet("/content/drive/MyDrive/default_train_0000.parquet", engine="pyarrow")
dataset3 = pd.read_parquet("/content/drive/MyDrive/default_validation_0000.parquet", engine="pyarrow")

# Tokenize the data
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tdataset2_text = tokenizer(dataset2["text"].tolist(), padding=True, truncation=True, max_length=128)
tdataset3_text = tokenizer(dataset3["text"].tolist(), padding=True, truncation=True, max_length=128)

# Create PyTorch tensors
labels2 = torch.tensor(dataset2['label'].values)
labels3 = torch.tensor(dataset3['label'].values)

input_ids2 = torch.tensor(tdataset2_text['input_ids'])
token_type_ids2 = torch.tensor(tdataset2_text['token_type_ids'])
attention_mask2 = torch.tensor(tdataset2_text['attention_mask'])

input_ids3 = torch.tensor(tdataset3_text['input_ids'])
token_type_ids3 = torch.tensor(tdataset3_text['token_type_ids'])
attention_mask3 = torch.tensor(tdataset3_text['attention_mask'])

from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self, input_ids, token_type_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.token_type_ids = token_type_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'token_type_ids': self.token_type_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }

# Create dataset instances
tensor_dataset2 = MyDataset(input_ids2, token_type_ids2, attention_mask2, labels2)
tensor_dataset3 = MyDataset(input_ids3, token_type_ids3, attention_mask3, labels3)


# Create PyTorch datasets
#tensor_dataset2 = TensorDataset(input_ids2, token_type_ids2, attention_mask2, labels2)
#tensor_dataset3 = TensorDataset(input_ids3, token_type_ids3, attention_mask3, labels3)

# Create DataLoaders
BATCH_SIZE = 8
data_loader2 = DataLoader(tensor_dataset2, batch_size=BATCH_SIZE, shuffle=True)
data_loader3 = DataLoader(tensor_dataset3, batch_size=BATCH_SIZE, shuffle=False)


# Define the custom Trainer class
class ImageDistilTrainer(Trainer):
    def __init__(self, student_model=None, teacher_model=None, temperature=1.0, *args, **kwargs):
        super().__init__(model=student_model, *args, **kwargs)
        self.teacher = teacher_model
        self.temperature = temperature

    def compute_loss(self, model, inputs, return_outputs=False):
        student_outputs = model(**inputs)
        with torch.no_grad():
            teacher_outputs = self.teacher(**inputs)

        # Calculate distillation loss using logits or predictions
        student_logits = student_outputs.logits / self.temperature
        teacher_logits = teacher_outputs.logits / self.temperature

        # Use predictions for distillation loss if num_labels is 1
        if student_outputs.logits.shape[1] == 1:
            loss = F.kl_div(
                F.logsigmoid(student_logits.squeeze()),
                torch.sigmoid(teacher_logits.squeeze()),
                reduction="batchmean"
            )
            # For binary classification, use BCEWithLogitsLoss
            loss += F.binary_cross_entropy_with_logits(
                student_logits.squeeze(),
                inputs['labels'].float()
            )
        else:
            loss = F.kl_div(F.log_softmax(student_logits, dim=-1),
                            F.softmax(teacher_logits, dim=-1),
                            reduction="batchmean")
            # For multi-class classification, use cross_entropy
            loss += F.cross_entropy(student_logits, inputs['labels'])

        return (loss, student_outputs) if return_outputs else loss

# Load the models
config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=1)
student_model = AutoModelForSequenceClassification.from_config(config)
teacher_model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', config=config)

# Freeze teacher model
for param in teacher_model.parameters():
    param.requires_grad = False


# Define training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/output",
    num_train_epochs=3,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    logging_dir="/content/drive/MyDrive/logging",
    logging_strategy="epoch",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    fp16=True,
)


print(type(tensor_dataset2[1]))
print(tensor_dataset2[1])
trainer = ImageDistilTrainer(
    student_model=student_model,
    teacher_model=teacher_model,
    train_dataset=tensor_dataset2,
    eval_dataset=tensor_dataset3,
    args=training_args,
    temperature=10.0,
)

trainer.train()

ValueError: mount failed

In [ ]:
j=8

In [ ]:
Chat history


ChatGPT
ChatGPT
GPT
Humanize AI
GPT
AI Humanizer

Explore GPTs
Yesterday
BERT Model Data Processing

Chemical vs Physical Properties
Chemical vs Physical Changes
Aws Discussion on Iceberg
Out of Office Message
Previous 7 Days
Follow-Up Email Request
Array Slice Error
Concurrent Validity vs Reliability
Form Submission Issue
Locating Data Engine Logs
Databricks Azure Migration Query
Protein Content Inquiry
Fridge Replacement Check Update
Cost Report Clarification
Access Restoration and Permissions
AAD Group Access Improvement
Previous 30 Days
Cost of Data Ingestion
Sentiment Model Help
Face-Framing Layers Tips
Time Off Request Notification
Time Off Request Update
Out-of-Office Notification
Study Spots After 5
Hypothalamus Functions Overview
Anonymous File Upload Methods
Chyme CCK Role
Protein and Muscle Building
Patching Schedule Request
Add Team workspace
Collaborate on a Team plan
ChatGPT 4o


User
You said:
import tf_keras as tfk
import gensim.downloader as api
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
from tf_keras import models as md
from tf_keras import layers as ly
from tf_keras import losses as ls
from tf_keras import optimizers as oz
from tf_keras import metrics as ms
import numpy as np
import tensorflow as tf
# import drive
# import files
# from datasets import Dataset
from transformers import (BertTokenizerFast, TFBertTokenizer, BertTokenizer, RobertaTokenizerFast,
                          DataCollatorWithPadding, TFRobertaForSequenceClassification, TFBertForSequenceClassification,
                          TFBertModel, create_optimizer)
import torch, pathlib, os, re, string, time
from torch import tensor
import pandas as pd
import transformers as trf
from transformers import AutoConfig

from google.colab import drive

import accelerate
from tf_keras.src.losses import kl_divergence
from transformers import Trainer, TrainingArguments
from tf_keras.src.optimizers.adam import Adam

import pickle

from tf_keras.src.optimizers.schedules.learning_rate_schedule import CosineDecay

import torch
from torch.utils.data import TensorDataset


from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
dataset2 = pd.read_parquet(r"/content/drive/MyDrive/default_train_0000.parquet", engine="pyarrow")
dataset3 = pd.read_parquet(r"/content/drive/MyDrive/default_validation_0000.parquet", engine="pyarrow")

cdataset2 = dataset2
dataset2_text = dataset2["text"].tolist()
dataset3_text = dataset3["text"].tolist()
cdataset3 = dataset3
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

tokenizer = BertTokenizerFast.from_pretrained(model_max_length=1024, pretrained_model_name_or_path='bert-base-uncased')
config = AutoConfig.from_pretrained(pretrained_model_name_or_path='bert-base-uncased')
tdataset2_text = tokenizer(dataset2_text, padding=True, truncation=True, max_length=128)
tdataset3_text = tokenizer(dataset3_text, padding=True, truncation=True, max_length=128)
tdataset2_dict = tdataset2_text.data
tdataset3_dict = tdataset3_text.data
cTdataset2 = pd.DataFrame.from_dict(tdataset2_dict)

cdataset2 = pd.concat([cdataset2, cTdataset2], axis=1)
tokenized_dataset2 = cdataset2

cTdataset3 = pd.DataFrame.from_dict(tdataset3_dict)

cdataset3 = pd.concat([cdataset3, cTdataset3], axis=1)
tokenized_dataset3 = cdataset3

print(cdataset2.shape)
print(cdataset2.columns)
print(len(cdataset2['input_ids']), len(cdataset2['token_type_ids']), len(cdataset2['attention_mask']), len(cdataset2['label']))
print(cdataset3.shape)
print(cdataset3.columns)
print(len(cdataset3['input_ids']), len(cdataset3['token_type_ids']), len(cdataset3['attention_mask']), len(cdataset3['label']))



BATCH_SIZE=8
numeric_feature_ames2 = ['input_ids', 'token_type_ids', 'attention_mask', 'label']
numeric_features2 = tokenized_dataset2[numeric_feature_names2]

numeric_feature_names3 = ['input_ids', 'token_type_ids', 'attention_mask', 'label']
numeric_features3 = tokenized_dataset3[numeric_feature_names3]

nf2l = numeric_features2["label"].values.tolist()
nf3l = numeric_features3["label"].values.tolist()
labels2 = torch.tensor(nf2l)
labels3 = torch.tensor(nf3l)

print(labels2.shape, labels2.size())
print(labels3.shape, labels3.size())

import torch.nn.utils.rnn as rnn


# Pad the lists to a fixed length
max_length = 128  # adjust this to the maximum length of your lists
padded_numeric_features2_id = rnn.pad_sequence([torch.tensor(x) for x in numeric_features2['input_ids']],
                                               batch_first=True)  # , maxlen=max_length)
padded_numeric_features3_id = rnn.pad_sequence([torch.tensor(x) for x in numeric_features3['input_ids']],
                                               batch_first=True)  # , maxlen=max_length)
padded_numeric_features2_ids = rnn.pad_sequence([torch.tensor(x) for x in numeric_features2['token_type_ids']],
                                                batch_first=True)  # , maxlen=max_length)
padded_numeric_features3_ids = rnn.pad_sequence([torch.tensor(x) for x in numeric_features3['token_type_ids']],
                                                batch_first=True)  # , maxlen=max_length)
padded_numeric_features2_mask = rnn.pad_sequence([torch.tensor(x) for x in numeric_features2['attention_mask']],
                                                 batch_first=True)  # , maxlen=max_length)
padded_numeric_features3_mask = rnn.pad_sequence([torch.tensor(x) for x in numeric_features3['attention_mask']],
                                                 batch_first=True)  # , maxlen=max_length)
# Convert the padded tensors to PyTorch tensors
import torch.nn.functional as F

padded_numeric_features2_id = F.pad(padded_numeric_features2_id, (0, max_length - padded_numeric_features2_id.shape[1]))
padded_numeric_features3_id = F.pad(padded_numeric_features3_id, (0, max_length - padded_numeric_features3_id.shape[1]))
padded_numeric_features2_ids = F.pad(padded_numeric_features2_ids,
                                     (0, max_length - padded_numeric_features2_ids.shape[1]))
padded_numeric_features3_ids = F.pad(padded_numeric_features3_ids,
                                     (0, max_length - padded_numeric_features3_ids.shape[1]))
padded_numeric_features2_mask = F.pad(padded_numeric_features2_mask,
                                      (0, max_length - padded_numeric_features2_mask.shape[1]))
padded_numeric_features3_mask = F.pad(padded_numeric_features3_mask,
                                      (0, max_length - padded_numeric_features3_mask.shape[1]))


print(padded_numeric_features2_id.shape, padded_numeric_features2_ids.shape, padded_numeric_features2_mask.shape, labels2.shape)
print(padded_numeric_features3_id.size(), padded_numeric_features3_ids.size(), padded_numeric_features3_mask.size(), labels3.size())
#print(tensor_dataset3.get_shape())
#print(type((padded_numeric_features2_id), type(padded_numeric_features2_ids), type( padded_numeric_features2_mask), type(labels2)))
print(type(padded_numeric_features3_id), type(padded_numeric_features3_ids), type(padded_numeric_features3_mask), type(labels3))
#tensor_dataset3 = TensorDataset(padded_numeric_features3_id, padded_numeric_features3_ids, padded_numeric_features3_mask, labels3)
#print(tensor_dataset3['input_ids'].size(), tensor_dataset3['token_type_ids'].size(), tensor_dataset3['attention_mask'].size(), tensor_dataset3['label'].size())
#print(tensor_dataset3.get_shape())

from torch.utils.data import DataLoader

tensor_dataset3 = TensorDataset(padded_numeric_features3_id, padded_numeric_features3_ids, padded_numeric_features3_mask, labels3)

data_loader3 = DataLoader(tensor_dataset3, batch_size=BATCH_SIZE)



# Get the shape and size of each tensor in the dataset
print(tensor_dataset3.tensors[0].shape)  # shape of padded_numeric_features3_id
print(tensor_dataset3.tensors[0].size())  # size of padded_numeric_features3_id

print(tensor_dataset3.tensors[1].shape)  # shape of padded_numeric_features3_ids
print(tensor_dataset3.tensors[1].size())  # size of padded_numeric_features3_ids

print(tensor_dataset3.tensors[2].shape)  # shape of padded_numeric_features3_mask
print(tensor_dataset3.tensors[2].size())  # size of padded_numeric_features3_mask

print(tensor_dataset3.tensors[3].shape)  # shape of labels3
print(tensor_dataset3.tensors[3].size())  # size of labels3
# Create a pandas DataFrame from the DataLoader
df3 = pd.DataFrame([batch for batch in data_loader3])

# Assign column names
df3.columns = ['input_ids', 'token_type_ids', 'attention_mask', 'label']




import torch
from torch.utils.data import Dataset, DataLoader

#print(len(labels2), len(input_ids2), len(attention_mask2), len(token_type_ids2))
print(len(labels3), len(padded_numeric_features3_id), len(padded_numeric_features3_ids), len(padded_numeric_features3_mask))

class CustomTensorDataset(Dataset):
    def __init__(self, input_ids, token_type_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.token_type_ids = token_type_ids
        self.attention_mask = attention_mask
        self.labels = labels
        self.field_names = ['input_ids', 'token_type_ids', 'attention_mask', 'labels']

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'token_type_ids': self.token_type_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }


# Assuming df is your pandas DataFrame
input_ids3 = [tensor for tensor in df3['input_ids']]
token_type_ids3 = [tensor for tensor in df3['token_type_ids']]
attention_mask3 = [tensor for tensor in df3['attention_mask']]
labels3 = df3['label'].values

#print(input_ids3.shape, token_type_ids3.shape, padded_numeric_features3_ids.shape, padded_numeric_features3_mask.shape, len(labels3))
print(len(input_ids3), len(token_type_ids3), len(padded_numeric_features3_ids), len(padded_numeric_features3_mask), len(labels3))


try:
    datasetxxx3 = CustomTensorDataset(input_ids3, token_type_ids3, attention_mask3, labels3)
    data_loaderxx3 = DataLoader(datasetxxx3, batch_size=8, shuffle=True)
except Exception as e:
    print(f"Error creating datasetxxx3: {e}")

tensor_dataset2 = TensorDataset(padded_numeric_features2_id, padded_numeric_features2_ids,
                                padded_numeric_features2_mask, labels2)
from torch.utils.data import DataLoader

data_loader2 = DataLoader(tensor_dataset2, batch_size=BATCH_SIZE)





# Create a pandas DataFrame from the DataLoader
df2 = pd.DataFrame([batch for batch in data_loader2])

# Assign column names
df2.columns = ['input_ids', 'token_type_ids', 'attention_mask', 'label']

import torch
from torch.utils.data import Dataset, DataLoader


class CustomTensorDataset(Dataset):
    def __init__(self, input_ids, token_type_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.token_type_ids = token_type_ids
        self.attention_mask = attention_mask
        self.labels = labels
        self.field_names = ['input_ids', 'token_type_ids', 'attention_mask', 'labels']

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'token_type_ids': self.token_type_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }


# Assuming df is your pandas DataFrame
input_ids2 = [tensor for tensor in df2['input_ids']]
token_type_ids2 = [tensor for tensor in df2['token_type_ids']]
attention_mask2 = [tensor for tensor in df2['attention_mask']]
labels2 = df2['label'].values

#print(input_ids3.shape, token_type_ids3.shape, padded_numeric_features3_ids.shape, padded_numeric_features3_mask.shape, len(labels3))
print(len(input_ids2), len(token_type_ids2), len(padded_numeric_features3_ids), len(padded_numeric_features3_mask), len(labels2))

try:
    datasetxxx2 = CustomTensorDataset(input_ids2, token_type_ids2, attention_mask2, labels2)
    data_loaderxx2 = DataLoader(datasetxxx2, batch_size=8, shuffle=True)
except Exception as e:
    print(f"Error creating datasetxxx2: {e}")

"""
datasetxxx2 = CustomTensorDataset(input_ids2, token_type_ids2, attention_mask2, labels2)

# Create a DataLoader from the custom dataset
data_loaderxx2 = DataLoader(datasetxxx2, batch_size=8, shuffle=True)"""

print(df2.shape, df2.columns)
print(df3.shape, df3.columns)


# model = AutoModelForSequenceClassification.from_pretrained(pretrained_model_name_or_path='bert-base-uncased',num_labels=1)
# model = AutoModelForSequenceClassification.from_scratch(num_labels=1)
# model = AutoModelForSequenceClassification(config=AutoConfig(num_labels=1))

# model = AutoModelForSequenceClassification.from_pretrained(pretrained_model_name_or_path='bert-base-uncased', num_labels=1)

#config = AutoConfig.from_pretrained(pretrained_model_name_or_path='bert-base-uncased')
config.num_labels = 1

model = AutoModelForSequenceClassification.from_config(config)
model.init_weights()
for param in model.bert.parameters():
    param.requires_grad = False

model.classifier.weight.requires_grad = True
model.classifier.bias.requires_grad = True
"""

print(len(input_ids2), len(token_type_ids2), len(attention_mask2), len(labels2))
datasetxxx2 = CustomTensorDataset(input_ids2, token_type_ids2, attention_mask2, labels2)
"""
from transformers import TrainingArguments, Trainer
import torch
import torch.nn as nn
import torch.nn.functional as F



class ImageDistilTrainer(Trainer):
    def __init__(self, student_model=None, teacher_model=None, train_dataset=None, eval_dataset=None, temperature=None, compute_metrics=None, *args, **kwargs):
        super().__init__(model=student_model, *args, **kwargs)
        self.teacher = teacher_model
        self.student = student_model
        # self.loss_function = nn.KLDivLoss(reduction="batchmean")
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.teacher.to(device)
        self.teacher.eval()
        self.temperature = temperature
        #teacher_model = teacher_model,  # the model to be trained
        #student_model=student_model,
        #args = training_args,  # training arguments
        self.train_dataset = train_dataset,  # training dataset
        self.eval_dataset = eval_dataset,  # evaluation dataset
        self.compute_metrics = compute_metrics


    def get_logits(inputs):
        teacher_inputs = {'input_ids': inputs['input_ids'], 'attention_mask': inputs['attention_mask']}
        student_inputs = {'input_ids': inputs['input_ids'], 'attention_mask': inputs['attention_mask']}
        teacher_logits = teacher_model(**teacher_inputs).logits
        student_logits = student_model(**student_inputs).logits
        return teacher_logits, student_logits


    def distillation_loss(teacher_logits, student_logits, temperature=10.0):
        teacher_probs = tf.nn.softmax(teacher_logits / temperature, axis=-1)
        student_probs = tf.nn.softmax(student_logits / temperature, axis=-1)
        return kl_divergence()(teacher_probs, student_probs)


    def compute_loss(model, inputs):
        teacher_logits, student_logits = get_logits(inputs)
        loss = distillation_loss(teacher_logits, student_logits)
        return loss





# ===============================================================================================================================

# from transformers import AutoModelForImageClassification, MobileNetV2Config, MobileNetV2ForImageClassification

training_args = TrainingArguments(
    output_dir=r"/content/drive/MyDrive",
    num_train_epochs=3,
    fp16=True,
    logging_dir=r"/content/drive/MyDrive",
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    # total # of training epochs
    # prediction_loss_only=compute_loss,
    per_device_train_batch_size=8,  # batch size
    # per device during training
    per_device_eval_batch_size=8,  # batch size for evaluation
    warmup_steps=500,  # number of warmup steps for learning rate scheduler
    weight_decay=0.01,
)

# =============================================================================================================================

num_epochs = 3
batches_per_epoch = 1  # 227(1821)[07196]69170,8
total_train_steps = int(batches_per_epoch * num_epochs)
# optimizer = Adam(init_lr=2e-5)#, num_warmup_steps=0, num_train_steps=total_train_steps)
optimizer = Adam()
scheduler = CosineDecay(initial_learning_rate=2e-5, decay_steps=total_train_steps)

####
from transformers import DataCollatorWithPadding, DataCollator


import torch
from transformers import AutoModelForSequenceClassification, AutoConfig, TrainingArguments, Trainer
from torch.utils.data import Dataset, DataLoader


# Define the custom data collator function
def custom_data_collator(features):
    input_ids = torch.tensor([f['input_ids'] for f in features])
    token_type_ids = torch.tensor([f['token_type_ids'] for f in features])
    attention_mask = torch.tensor([f['attention_mask'] for f in features])
    labels = torch.tensor([f['labels'] for f in features])

    return {
        'input_ids': input_ids,
        'token_type_ids': token_type_ids,
        'attention_mask': attention_mask,
        'labels': labels
    }

# Define the ImageDistilTrainer class

# Define the training arguments
training_args = TrainingArguments(
    output_dir=r"/content/drive/MyDrive",
    #num_train_epochs
	  num_train_epochs=3,
    fp16=True,
    logging_dir=r"/content/drive/MyDrive",
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
)

# Load the pre-trained BERT model and tokenizer
#tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
"""
config = AutoConfig.from_pretrained('bert-base-uncased')
config.num_labels = 1
model = AutoModelForSequenceClassification.from_config(config)
model.init_weights()
for param in model.bert.parameters():
    param.requires_grad = False
model.classifier.weight.requires_grad = True
model.classifier.bias.requires_grad = True
"""
# Create the ImageDistilTrainer instance
trainer = ImageDistilTrainer(
    student_model=model,
    teacher_model=model,
    train_dataset=datasetxxx2,
    eval_dataset=datasetxxx3,
    data_collator=custom_data_collator,
    compute_metrics=lambda pred: {'accuracy': torch.sum(pred.label_ids == pred.predictions.argmax(-1))},
    temperature=10,
)

# Train the model
trainer.train()

#################################the above is my code for knowledge distilaation bert to bert (self), now waht should be tha appropriate code, taht no erros is faced.
ChatGPT said:
ChatGPT
Your code involves setting up a knowledge distillation framework where a student BERT model is trained by distilling knowledge from a teacher BERT model. However, there are a few issues and best practices that can help improve your implementation and avoid errors. Here's an appropriate version of your code, cleaned up and organized for better clarity and functionality:

python
Copy code
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoConfig, Trainer, TrainingArguments)

# Load the data
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
dataset2 = pd.read_parquet("/content/drive/MyDrive/default_train_0000.parquet", engine="pyarrow")
dataset3 = pd.read_parquet("/content/drive/MyDrive/default_validation_0000.parquet", engine="pyarrow")

# Tokenize the data
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tdataset2_text = tokenizer(dataset2["text"].tolist(), padding=True, truncation=True, max_length=128)
tdataset3_text = tokenizer(dataset3["text"].tolist(), padding=True, truncation=True, max_length=128)

# Create PyTorch tensors
labels2 = torch.tensor(dataset2['label'].values)
labels3 = torch.tensor(dataset3['label'].values)

input_ids2 = torch.tensor(tdataset2_text['input_ids'])
token_type_ids2 = torch.tensor(tdataset2_text['token_type_ids'])
attention_mask2 = torch.tensor(tdataset2_text['attention_mask'])

input_ids3 = torch.tensor(tdataset3_text['input_ids'])
token_type_ids3 = torch.tensor(tdataset3_text['token_type_ids'])
attention_mask3 = torch.tensor(tdataset3_text['attention_mask'])

# Create PyTorch datasets
tensor_dataset2 = TensorDataset(input_ids2, token_type_ids2, attention_mask2, labels2)
tensor_dataset3 = TensorDataset(input_ids3, token_type_ids3, attention_mask3, labels3)

# Create DataLoaders
BATCH_SIZE = 8
data_loader2 = DataLoader(tensor_dataset2, batch_size=BATCH_SIZE, shuffle=True)
data_loader3 = DataLoader(tensor_dataset3, batch_size=BATCH_SIZE, shuffle=False)

# Define the custom Trainer class
class ImageDistilTrainer(Trainer):
    def __init__(self, student_model=None, teacher_model=None, temperature=1.0, *args, **kwargs):
        super().__init__(model=student_model, *args, **kwargs)
        self.teacher = teacher_model
        self.temperature = temperature

    def compute_loss(self, model, inputs, return_outputs=False):
        student_outputs = model(**inputs)
        with torch.no_grad():
            teacher_outputs = self.teacher(**inputs)

        # Calculate distillation loss
        student_logits = student_outputs.logits / self.temperature
        teacher_logits = teacher_outputs.logits / self.temperature
        loss = F.kl_div(F.log_softmax(student_logits, dim=-1), F.softmax(teacher_logits, dim=-1), reduction="batchmean")

        # Add cross-entropy loss
        loss += F.cross_entropy(student_logits, inputs['labels'])

        return (loss, student_outputs) if return_outputs else loss

# Load the models
config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=1)
student_model = AutoModelForSequenceClassification.from_config(config)
teacher_model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', config=config)

# Freeze teacher model
for param in teacher_model.parameters():
    param.requires_grad = False

# Define training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/output",
    num_train_epochs=3,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    logging_dir="/content/drive/MyDrive/logging",
    logging_strategy="epoch",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
)

# Initialize the Trainer
trainer = ImageDistilTrainer(
    student_model=student_model,
    teacher_model=teacher_model,
    train_dataset=tensor_dataset2,
    eval_dataset=tensor_dataset3,
    data_collator=None,  # Default collator works fine
    args=training_args,
    temperature=10.0,
)

# Train the model
trainer.train()
Key Adjustments:
Correct compute_loss Implementation: The compute_loss method correctly implements the distillation loss by calculating the KL divergence between the teacher and student logits and adding cross-entropy loss.

Simplified Dataset Handling: The TensorDataset and DataLoader structures are simplified and directly created from the tokenized datasets.

Model and Training Setup: The teacher model's weights are frozen to ensure that only the student model is trained. The configuration for training is simplified with TrainingArguments.

Custom Trainer Class: A custom Trainer class (ImageDistilTrainer) is used to handle the specific logic required for knowledge distillation.

This code should run without errors, provided that the necessary packages are correctly installed and the environment is properly set up.






You said:
i am trying to do sentiment anlysisi, now what should be my problem statement
ChatGPT said:
ChatGPT
Memory updated
For a sentiment analysis project, here’s a well-defined problem statement you can use:

Problem Statement: The goal of this project is to develop a sentiment analysis model capable of classifying textual data into different sentiment categories (e.g., positive, negative, neutral). The model will be trained on labeled text datasets and should generalize well to unseen data. The key objective is to accurately identify the sentiment expressed in user-generated content, such as reviews, social media posts, or customer feedback, in order to facilitate decision-making and improve user experience.

You can adjust this problem statement depending on your dataset and specific objectives, such as targeting product reviews, tweets, or other forms of communication.






You said:
i am a novice and beginener in sentimentanlysis, now which probleme stattemnt , should be ther, which is needed in today's time
ChatGPT said:
ChatGPT
Memory updated
As a beginner in sentiment analysis, it’s good to start with a problem statement that is both relevant to today’s needs and simple enough to help you learn the basics. Here's a practical and modern problem statement you can use:

Problem Statement: The objective of this project is to build a sentiment analysis model that can classify customer reviews into positive, negative, or neutral categories. The model will help businesses understand customer satisfaction and identify areas for improvement. Using customer reviews from platforms such as e-commerce sites, the model aims to accurately assess the sentiment to assist companies in making data-driven decisions to enhance product quality and customer experience.

This type of problem is highly relevant because businesses today heavily rely on user feedback and social media sentiment to guide their strategies. It will also give you exposure to real-world challenges while working with sentiment analysis.

Once you get comfortable, you can explore more complex datasets and tasks like analyzing sentiments on social media posts or assessing brand reputation.






You said:
so, should i serach that model on huggingface or , select a good model, and perform fine /tuning or knowledge distillation, or waht are other approcaehes, so taht my model result can improve
ChatGPT said:
ChatGPT
To improve your sentiment analysis model as a beginner, you can approach the task in a few different ways. Let’s break down the best strategies for you:

1. Using Pre-Trained Models (Hugging Face)
Best Practice: Start with pre-trained models from Hugging Face, as it will save time and provide good baselines. You can search for sentiment analysis models like BERT, RoBERTa, or DistilBERT, which are already trained on large text corpora.

Approach:

Search for models fine-tuned on sentiment tasks (such as bert-base-uncased, distilbert-base-uncased, etc.).
Fine-Tune the Model: After selecting a model, fine-tune it on your specific dataset (e.g., product reviews, social media posts). Fine-tuning involves training the last few layers of a pre-trained model to adapt it to your data.
Steps:

Search for pre-trained models on Hugging Face.
Load the model and tokenizer.
Fine-tune it using your dataset (usually takes fewer epochs than training from scratch).
2. Knowledge Distillation
When to Use: If you need a smaller and faster model (for deployment on low-resource environments like mobile apps), knowledge distillation is a great technique. Here, a large “teacher” model (like BERT) teaches a smaller “student” model.

Approach:

Fine-tune a large model (like BERT) on your dataset and use it to guide a smaller model (like DistilBERT) using knowledge distillation.
The smaller model tries to mimic the outputs of the larger model while learning the task.
Steps:

Fine-tune a large model (teacher).
Distill its knowledge into a smaller model (student) by minimizing the difference between their outputs (logits).
3. Transfer Learning (Pretrained Transformers + Fine-Tuning)
Why It Works: Most state-of-the-art results in NLP today come from transfer learning using large models (like BERT, GPT, etc.).

Approach:

Take a large pre-trained model and fine-tune it on your dataset.
Experiment with different pre-trained models (BERT, RoBERTa, DistilBERT, etc.) to see which performs best on your specific data.
Steps:

Load a pre-trained transformer model.
Fine-tune on your labeled sentiment dataset.
Evaluate and compare performance across models.
4. Hyperparameter Tuning
Why It’s Important: Once you’ve selected a model, optimizing the hyperparameters (like learning rate, batch size, dropout, etc.) can significantly improve performance.

Approach:

Use libraries like Optuna or Ray to automate hyperparameter tuning.
Experiment with different batch sizes, learning rates, optimizers (e.g., AdamW, SGD), and epochs.
Steps:

Start with default parameters.
Use hyperparameter tuning tools to find the best combination.
5. Data Augmentation
Why It Works: Improving data quality can also lead to better results. If you have a small dataset, using data augmentation techniques can create new samples and improve generalization.

Approach:

Synonym replacement, back-translation, or using paraphrasing tools can help create additional training data.
Steps:

Apply simple augmentation techniques to your dataset.
Retrain the model with the augmented data.
6. Ensemble Learning
Why It’s Effective: Combining predictions from multiple models often improves performance. Ensemble methods can help smooth out the biases of individual models.

Approach:

Train multiple models (e.g., BERT, RoBERTa, etc.) and combine their predictions using techniques like averaging, majority voting, or stacking.
Steps:

Train multiple models.
Combine their outputs for final predictions.
Next Steps:
Begin with a pre-trained model on Hugging Face, as it’s beginner-friendly and allows you to focus on fine-tuning without worrying about training from scratch.
Experiment with fine-tuning, followed by hyperparameter tuning and knowledge distillation for deployment in low-resource environments.
Once you’re comfortable, explore data augmentation and ensemble learning for even better performance.
Would you like help in starting with a specific model from Hugging Face, or do you want to explore more about hyperparameter tuning or distillation?






You said:
starting specifiinc model in huggingace hub
ChatGPT said:
ChatGPT
Here’s a step-by-step guide to getting started with a specific model from Hugging Face Hub, such as BERT for sentiment analysis:

1. Install the Necessary Libraries
Make sure you have the transformers and datasets libraries installed.

bash
Copy code
pip install transformers datasets
2. Choose a Pre-trained Model
For sentiment analysis, you can start with models like bert-base-uncased, distilbert-base-uncased, or models fine-tuned specifically for sentiment tasks.

A popular one for sentiment analysis is:

nlptown/bert-base-multilingual-uncased-sentiment
This model can classify text into 5 sentiment categories (from very negative to very positive).

3. Load the Pre-trained Model and Tokenizer
Here’s how to load the model and tokenizer:

python
Copy code
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained("nlptown/bert-base-multilingual-uncased-sentiment")

# Load the pre-trained model
model = AutoModelForSequenceClassification.from_pretrained("nlptown/bert-base-multilingual-uncased-sentiment")
4. Tokenize Your Input Text
You’ll need to prepare the text for the model by tokenizing it.

python
Copy code
# Example text to analyze sentiment
text = "I love this product! It’s amazing."

# Tokenize the text
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
5. Perform Sentiment Analysis
Pass the tokenized input to the model and retrieve the predicted sentiment.

python
Copy code
# Perform inference
outputs = model(**inputs)

# Get the predicted sentiment class (logits)
logits = outputs.logits

# Apply softmax to get probabilities for each sentiment label
import torch
probs = torch.nn.functional.softmax(logits, dim=-1)

# Get the predicted sentiment class
predicted_class = torch.argmax(probs, dim=-1)

# Print the sentiment score and predicted label
sentiment_labels = ['very negative', 'negative', 'neutral', 'positive', 'very positive']
print(f'Sentiment: {sentiment_labels[predicted_class]}')
6. Fine-Tuning the Model (Optional)
Once you are familiar with how to use the model, you may want to fine-tune it on your own dataset.

Here’s a brief outline on how to fine-tune the model:

Prepare Your Dataset: Load your dataset using datasets or pandas.
Tokenize the Dataset: Use the tokenizer to preprocess the dataset.
Define the Trainer: Use the Hugging Face Trainer API to handle the training loop.
python
Copy code
from datasets import load_dataset
from transformers import Trainer, TrainingArguments

# Load dataset (example: IMDb dataset)
dataset = load_dataset("imdb")

# Tokenize the dataset
def preprocess_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True)

tokenized_datasets = dataset.map(preprocess_function, batched=True)

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
)

# Define the trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
)

# Fine-tune the model
trainer.train()
7. Evaluate and Save the Model
After fine-tuning, you can evaluate the model on test data:

python
Copy code
# Evaluate the model
trainer.evaluate()

# Save the fine-tuned model
model.save_pretrained("./fine-tuned-model")
tokenizer.save_pretrained("./fine-tuned-model")
Summary of Steps:
Install necessary libraries.
Load a pre-trained model and tokenizer from Hugging Face Hub.
Tokenize input text and make predictions using the model.
Fine-tune the model on your specific dataset if needed.
Evaluate and save the model.
This process gives you a strong foundation for sentiment analysis with state-of-the-art models. Let me know if you need help with any specific part, like loading data or fine-tuning!






You said:
but then in 3red point, ,the acuay comes lowk iff ill trainted to full epch
ChatGPT said:
Which response do you prefer?
Your choice will help make ChatGPT better.
ChatGPT
Response 1
When fine-tuning a pre-trained model, it's common to see low accuracy at first or after training for a full epoch, especially if the model is not well-optimized or if it’s overfitting/underfitting. Here are some strategies to improve accuracy when fine-tuning:

1. Early Stopping
Why: Fine-tuning for too many epochs can lead to overfitting, where the model performs well on training data but poorly on test data. Implementing early stopping ensures that training stops when the model’s performance starts to degrade.
Solution: Monitor validation loss or accuracy during training and stop if there’s no improvement after a certain number of steps.
python
Copy code
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="steps",
    eval_steps=500,  # Evaluate every 500 steps
    save_steps=500,
    load_best_model_at_end=True,
    num_train_epochs=3,
    early_stopping_patience=2,  # Stop if no improvement after 2 evaluations
    save_total_limit=1,  # Save only the best model
)
2. Smaller Learning Rate
Why: Pre-trained models have already learned a lot from large datasets. Using a large learning rate can cause drastic updates to the weights, leading to lower accuracy.
Solution: Use a smaller learning rate when fine-tuning. A typical range for fine-tuning is between 1e-5 and 5e-5.
python
Copy code
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,  # Small learning rate
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    evaluation_strategy="epoch",
    save_total_limit=1,  # Save only the best model
)
3. Freezing Some Layers
Why: Pre-trained models like BERT have many layers. Fine-tuning all layers on a small dataset may cause overfitting. Freezing the lower layers (which contain general language features) and fine-tuning only the top layers can improve accuracy.
Solution: Freeze lower layers of the model and only train the top few layers.
python
Copy code
# Freeze all BERT layers
for param in model.bert.parameters():
    param.requires_grad = False

# Fine-tune only the classifier
model.classifier.weight.requires_grad = True
model.classifier.bias.requires_grad = True
4. Data Augmentation
Why: If you have a small dataset, the model may not generalize well. Data augmentation techniques such as paraphrasing, back-translation, or adding noise can increase the size and diversity of your dataset.
Solution: Use data augmentation libraries to create more training data or techniques like synonym replacement to modify your text slightly.
5. Balanced Dataset
Why: If your dataset is imbalanced (e.g., too many positive reviews compared to negative ones), the model may get biased and predict the majority class more often, leading to low accuracy for minority classes.
Solution: Balance your dataset by:
Undersampling the majority class or oversampling the minority class.
Using techniques like class weights to give more importance to minority classes.
python
Copy code
# Example: Assign class weights to handle imbalance
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=[0, 1], y=labels)
class_weights = torch.tensor(class_weights, dtype=torch.float)

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
)
6. Batch Size
Why: Small batch sizes can cause the model to not learn well, as updates will be based on too little data, leading to instability. On the other hand, large batch sizes can make the model learn slowly.
Solution: Start with a reasonable batch size (8 or 16) and experiment with increasing it if memory allows.
7. Hyperparameter Tuning
Why: Fine-tuning performance can vary greatly based on hyperparameters like learning rate, batch size, number of epochs, and dropout rates.
Solution: Use hyperparameter optimization libraries like Optuna or Ray to search for the best hyperparameter configuration.
python
Copy code
# Example: Use Optuna for hyperparameter tuning
import optuna
from transformers import Trainer, TrainingArguments

def objective(trial):
    # Suggest hyperparameters
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 5e-5)
    num_train_epochs = trial.suggest_int('num_train_epochs', 2, 4)
    batch_size = trial.suggest_categorical('batch_size', [8, 16])

    training_args = TrainingArguments(
        output_dir="./results",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_train_epochs,
        evaluation_strategy="epoch",
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets['train'],
        eval_dataset=tokenized_datasets['test'],
    )

    result = trainer.train()
    return result.training_loss

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=10)
8. Regularization (Dropout, Weight Decay)
Why: Regularization techniques like dropout and weight decay help prevent overfitting, especially on small datasets.
Solution: Use dropout and weight decay in your training to improve generalization.
python
Copy code
training_args = TrainingArguments(
    output_dir="./results",
    weight_decay=0.01,  # Add weight decay to improve generalization
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    evaluation_strategy="epoch",
)
9. Learning Rate Schedulers
Why: A fixed learning rate may not always be the best approach. Using learning rate schedulers, like cosine annealing or linear warmup, helps the model converge faster.
Solution: Implement learning rate schedulers to adjust the learning rate over time.
python
Copy code
from transformers import get_scheduler

optimizer = AdamW(model.parameters(), lr=2e-5)
num_training_steps = len(train_dataloader) * num_train_epochs
lr_scheduler = get_scheduler(
    name="linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)
Conclusion:
Start small by using early stopping, freezing layers, or reducing the learning rate.
Experiment with hyperparameters like learning rate, batch size, and dropout.
Regularize and augment your data if the dataset is small or imbalanced.
By following these strategies, you can significantly improve the model's performance during fine-tuning. Let me know if you need further assistance with any of these techniques!


I prefer this response
ChatGPT
Response 2
If the accuracy in your sentiment analysis task is low after training for full epochs, here are some common reasons and strategies to address this issue:

1. Overfitting or Underfitting
Symptoms: If the model performs well on the training set but poorly on the validation/test set, it might be overfitting. If the model performs poorly on both, it could be underfitting.
Solutions:
Reduce model complexity: Use a simpler model like DistilBERT or add regularization like dropout.
Early Stopping: Use early stopping based on validation accuracy or loss. Stop training when the model stops improving.
Increase Data: If you have a small dataset, try adding more data or using data augmentation.
More Epochs: If the model is underfitting, you may need more epochs.
2. Learning Rate Too High/Low
Symptoms: If the model isn’t improving or is fluctuating during training, the learning rate may be inappropriate.
Solutions:
Start with a small learning rate like 1e-5 or 2e-5 for fine-tuning pre-trained models. You can also use a learning rate scheduler to dynamically adjust it during training.
3. Class Imbalance in Dataset
Symptoms: If one class (e.g., positive or negative) is over-represented in your dataset, the model might be biased toward that class.
Solutions:
Class Weights: Assign class weights when training to penalize under-represented classes more.
Oversampling/Undersampling: Balance your dataset by oversampling the minority class or undersampling the majority class.
Example of using class weights in the Trainer:

python
Copy code
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = {i: class_weights[i] for i in range(len(class_weights))}

# Pass this to the Trainer as an argument
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    class_weights=class_weights_dict
)
4. Batch Size Too Small/Too Large
Symptoms: Very small batch sizes can result in noisy gradients and slower training, while very large batches may not generalize well.
Solutions:
Experiment with different batch sizes. A good starting point for BERT models is usually a batch size of 8 or 16, depending on your GPU memory.
5. Insufficient Fine-Tuning
Symptoms: If the accuracy is still low after a full epoch of training, you may need to fine-tune the model longer or unfreeze more layers.
Solutions:
Unfreeze more layers: By default, fine-tuning only updates the classifier head. You can unfreeze more layers of the pre-trained model for deeper fine-tuning.
Example of unfreezing layers:

python
Copy code
for param in model.bert.parameters():
    param.requires_grad = True  # Unfreeze BERT layers
6. Use of the Wrong Pre-Trained Model
Symptoms: Some models are better suited for certain tasks or languages. If you're working with specific types of text (e.g., tweets, reviews), a generic pre-trained model might not be optimal.
Solutions:
Choose a model fine-tuned on sentiment analysis if possible (e.g., distilbert-base-uncased-finetuned-sst-2-english).
Try multiple models from Hugging Face and compare their performance.
7. Data Preprocessing Issues
Symptoms: Incorrect tokenization or preprocessing can lead to lower accuracy.
Solutions:
Text cleaning: Remove unnecessary characters, stopwords, and special tokens that may confuse the model.
Ensure that input text is preprocessed similarly to how the pre-trained model was trained (e.g., use the correct tokenizer).
8. Improper Validation Set
Symptoms: If validation accuracy fluctuates a lot, your validation set might not represent the real-world distribution.
Solutions:
Split the data correctly to ensure the train, validation, and test sets are representative.
Consider cross-validation for better evaluation.
9. Experiment with Different Loss Functions
Symptoms: If you suspect your loss function isn't capturing the task well, it could affect accuracy.
Solutions:
Stick with CrossEntropyLoss for classification tasks, but consider tweaking the loss weights if there's class imbalance.
10. Hyperparameter Tuning
Symptoms: Poor performance even after training could be due to improper hyperparameters.
Solutions:
Grid Search or Random Search: Automate hyperparameter tuning using tools like Optuna or Ray Tune to optimize learning rates, batch sizes, and epochs.
Example of hyperparameter search:

python
Copy code
from transformers import Trainer, TrainingArguments

def model_init():
    return AutoModelForSequenceClassification.from_pretrained("bert-base-uncased")

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
)

trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

best_run = trainer.hyperparameter_search(n_trials=10, direction="maximize")
General Process to Address Low Accuracy:
Start with a pre-trained model from Hugging Face and fine-tune on your dataset.
Experiment with different learning rates (1e-5, 2e-5 are good starting points).
Monitor validation accuracy and loss during training to identify overfitting or underfitting.
Use early stopping to prevent overfitting and save the best-performing model.
Unfreeze more layers of the model if necessary and fine-tune for longer.
By using these strategies, you can systematically improve your model's accuracy over time. Would you like assistance with setting up one of these approaches?


I prefer this response



Message ChatGPT

ChatGPT can make mistakes. Check important info.
?


In [ ]:
import datasets as ds

In [ ]:
pip show datasets

Name: datasets
Version: 2.21.0
Summary: HuggingFace community-driven open-source library of datasets
Home-page: https://github.com/huggingface/datasets
Author: HuggingFace Inc.
Author-email: thomas@huggingface.co
License: Apache 2.0
Location: /usr/local/lib/python3.10/dist-packages
Requires: aiohttp, dill, filelock, fsspec, huggingface-hub, multiprocess, numpy, packaging, pandas, pyarrow, pyyaml, requests, tqdm, xxhash
Required-by: 


In [ ]:
# Install necessary libraries
#!pip install transformers datasets torch

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
#from transformers import GlueDataset
from google.colab import drive#

drive.mount('/content/drive')
#dataset2 = pd.read_parquet("/content/drive/MyDrive/default_train_0000.parquet", engine="pyarrow")
# Load the dataset
dataset = load_dataset("glue", "mnli")
#dataset = GlueDataset("glue", "mnli")
# Load tokenizers
teacher_tokenizer = AutoTokenizer.from_pretrained("roberta-large")
student_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenize the dataset
def preprocess_function(examples):
    return teacher_tokenizer(examples['premise'], examples['hypothesis'], truncation=True, padding=True, max_length=128)

tokenized_datasets = dataset.map(preprocess_function, batched=True)

# Prepare the models
teacher_model = AutoModelForSequenceClassification.from_pretrained("roberta-large", num_labels=3)
student_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3)

# Freeze teacher model's parameters
for param in teacher_model.parameters():
    param.requires_grad = False

# Custom Trainer with knowledge distillation
class DistillationTrainer(Trainer):
    def __init__(self, teacher_model, *args, temperature=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher = teacher_model
        self.temperature = temperature
        self.teacher.eval()

    def compute_loss(self, model, inputs, return_outputs=False):
        student_outputs = model(**inputs)
        student_logits = student_outputs.logits

        with torch.no_grad():
            teacher_outputs = self.teacher(**inputs)
            teacher_logits = teacher_outputs.logits

        # Calculate the distillation loss using KL Divergence
        loss = F.kl_div(
            F.log_softmax(student_logits / self.temperature, dim=-1),
            F.softmax(teacher_logits / self.temperature, dim=-1),
            reduction="batchmean"
        ) * (self.temperature ** 2)

        # Adding cross-entropy loss
        loss += F.cross_entropy(student_logits, inputs["labels"])

        return (loss, student_outputs) if return_outputs else loss

# Training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
)

# Initialize the distillation trainer
trainer = DistillationTrainer(
    teacher_model=teacher_model,
    model=student_model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation_matched'],
    tokenizer=student_tokenizer,
    temperature=2.0  # Higher temperature smooths the probabilities from the teacher
)

# Train the student model
trainer.train()

# Evaluate the student model on the validation set
trainer.evaluate()

# Example for prediction
def predict_sentiment(text_premise, text_hypothesis):
    inputs = student_tokenizer(text_premise, text_hypothesis, return_tensors="pt", truncation=True, padding=True)
    outputs = student_model(**inputs)
    predicted_class = torch.argmax(outputs.logits, dim=-1).item()
    label_map = {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
    return label_map[predicted_class]

# Example usage
text_premise = "The dog is flying in the park."
text_hypothesis = "The dog is beyond death."
predicted_sentiment = predict_sentiment(text_premise, text_hypothesis)
print(f"Predicted sentiment: {predicted_sentiment}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating validation_matched split:   0%|          | 0/9815 [00:00<?, ? examples/s]

Generating validation_mismatched split:   0%|          | 0/9832 [00:00<?, ? examples/s]

...

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


IndexError: index out of range in self

In [ ]:
# Install necessary libraries
#!pip install transformers datasets torch

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
#from datasets import load_dataset
from transformers import GlueDataset
from google.colab import drive#

drive.mount('/content/drive')
#dataset2 = pd.read_parquet("/content/drive/MyDrive/default_train_0000.parquet", engine="pyarrow")
# Load the dataset
#dataset = load_dataset("glue", "mnli")
dataset = GlueDataset("glue", "mnli")
# Load tokenizers
teacher_tokenizer = AutoTokenizer.from_pretrained("roberta-large")
student_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenize the dataset
def preprocess_function(examples):
    return teacher_tokenizer(examples['premise'], examples['hypothesis'], truncation=True, padding=True, max_length=128)

tokenized_datasets = dataset.map(preprocess_function, batched=True)

# Prepare the models
teacher_model = AutoModelForSequenceClassification.from_pretrained("roberta-large", num_labels=3)
student_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3)

# Freeze teacher model's parameters
for param in teacher_model.parameters():
    param.requires_grad = False

# Custom Trainer with knowledge distillation
class DistillationTrainer(Trainer):
    def __init__(self, teacher_model, *args, temperature=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher = teacher_model
        self.temperature = temperature
        self.teacher.eval()

    def compute_loss(self, model, inputs, return_outputs=False):
        student_outputs = model(**inputs)
        student_logits = student_outputs.logits

        with torch.no_grad():
            teacher_outputs = self.teacher(**inputs)
            teacher_logits = teacher_outputs.logits

        # Calculate the distillation loss using KL Divergence
        loss = F.kl_div(
            F.log_softmax(student_logits / self.temperature, dim=-1),
            F.softmax(teacher_logits / self.temperature, dim=-1),
            reduction="batchmean"
        ) * (self.temperature ** 2)

        # Adding cross-entropy loss
        loss += F.cross_entropy(student_logits, inputs["labels"])

        return (loss, student_outputs) if return_outputs else loss

# Training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
)

# Initialize the distillation trainer
trainer = DistillationTrainer(
    teacher_model=teacher_model,
    model=student_model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation_matched'],
    tokenizer=student_tokenizer,
    temperature=2.0  # Higher temperature smooths the probabilities from the teacher
)

# Train the student model
trainer.train()

# Evaluate the student model on the validation set
trainer.evaluate()

# Example for prediction
def predict_sentiment(text_premise, text_hypothesis):
    inputs = student_tokenizer(text_premise, text_hypothesis, return_tensors="pt", truncation=True, padding=True)
    outputs = student_model(**inputs)
    predicted_class = torch.argmax(outputs.logits, dim=-1).item()
    label_map = {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
    return label_map[predicted_class]

# Example usage
text_premise = "The dog is flying in the park."
text_hypothesis = "The dog is beyond death."
predicted_sentiment = predict_sentiment(text_premise, text_hypothesis)
print(f"Predicted sentiment: {predicted_sentiment}")

Mounted at /content/drive


/usr/local/lib/python3.10/dist-packages/transformers/data/datasets/glue.py:88: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py
  warnings.warn(


AttributeError: 'str' object has no attribute 'task_name'

In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 11.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Uninstalling pyarrow-14.0.2:
      Successfully uninstalled pyarrow-14.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 requires pyarrow<15.0.0a0,>=14.0.1, but you have pyarrow 17.0.0 which is incompatible.
ibis-framework 8.0.0 requires pyarrow<16,>=2, but you have pyarrow 17.0.0 which is incompatible.


In [ ]:
j=7

In [ ]:
from transformers import GlueDataset

In [ ]:
import datasets

ModuleNotFoundError: No module named 'datasets'

In [ ]:
# Install necessary libraries
#!pip install transformers datasets torch

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
from google.colab import drive#

drive.mount('/content/drive')
#dataset2 = pd.read_parquet("/content/drive/MyDrive/default_train_0000.parquet", engine="pyarrow")
# Load the dataset
dataset = load_dataset("glue", "mnli")

# Load tokenizers
teacher_tokenizer = AutoTokenizer.from_pretrained("roberta-large")
student_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenize the dataset
def preprocess_function(examples):
    return teacher_tokenizer(examples['premise'], examples['hypothesis'], truncation=True, padding=True, max_length=128)

tokenized_datasets = dataset.map(preprocess_function, batched=True)

# Prepare the models
teacher_model = AutoModelForSequenceClassification.from_pretrained("roberta-large", num_labels=3)
student_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3)

# Freeze teacher model's parameters
for param in teacher_model.parameters():
    param.requires_grad = False

# Custom Trainer with knowledge distillation
class DistillationTrainer(Trainer):
    def __init__(self, teacher_model, *args, temperature=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher = teacher_model
        self.temperature = temperature
        self.teacher.eval()

    def compute_loss(self, model, inputs, return_outputs=False):
        student_outputs = model(**inputs)
        student_logits = student_outputs.logits

        with torch.no_grad():
            teacher_outputs = self.teacher(**inputs)
            teacher_logits = teacher_outputs.logits

        # Calculate the distillation loss using KL Divergence
        loss = F.kl_div(
            F.log_softmax(student_logits / self.temperature, dim=-1),
            F.softmax(teacher_logits / self.temperature, dim=-1),
            reduction="batchmean"
        ) * (self.temperature ** 2)

        # Adding cross-entropy loss
        loss += F.cross_entropy(student_logits, inputs["labels"])

        return (loss, student_outputs) if return_outputs else loss

# Training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
)

# Initialize the distillation trainer
trainer = DistillationTrainer(
    teacher_model=teacher_model,
    model=student_model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation_matched'],
    tokenizer=student_tokenizer,
    temperature=2.0  # Higher temperature smooths the probabilities from the teacher
)

# Train the student model
trainer.train()

# Evaluate the student model on the validation set
trainer.evaluate()

# Example for prediction
def predict_sentiment(text_premise, text_hypothesis):
    inputs = student_tokenizer(text_premise, text_hypothesis, return_tensors="pt", truncation=True, padding=True)
    outputs = student_model(**inputs)
    predicted_class = torch.argmax(outputs.logits, dim=-1).item()
    label_map = {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
    return label_map[predicted_class]

# Example usage
text_premise = "The dog is flying in the park."
text_hypothesis = "The dog is beyond death."
predicted_sentiment = predict_sentiment(text_premise, text_hypothesis)
print(f"Predicted sentiment: {predicted_sentiment}")


ValueError: mount failed

In [ ]:
# Install necessary libraries
#!pip install transformers datasets torch

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
from google.colab import drive#

drive.mount('/content/drive')
#dataset2 = pd.read_parquet("/content/drive/MyDrive/default_train_0000.parquet", engine="pyarrow")
# Load the dataset
dataset = load_dataset("glue", "mnli")

# Load tokenizers
teacher_tokenizer = AutoTokenizer.from_pretrained("roberta-large")
student_tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# Tokenize the dataset
def preprocess_function(examples):
    return teacher_tokenizer(examples['premise'], examples['hypothesis'], truncation=True, padding=True, max_length=128)

tokenized_datasets = dataset.map(preprocess_function, batched=True)

# Prepare the models
teacher_model = AutoModelForSequenceClassification.from_pretrained("roberta-large", num_labels=3)
student_model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3)

# Freeze teacher model's parameters
for param in teacher_model.parameters():
    param.requires_grad = False

# Custom Trainer with knowledge distillation
class DistillationTrainer(Trainer):
    def __init__(self, teacher_model, *args, temperature=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher = teacher_model
        self.temperature = temperature
        self.teacher.eval()

    def compute_loss(self, model, inputs, return_outputs=False):
        student_outputs = model(**inputs)
        student_logits = student_outputs.logits

        with torch.no_grad():
            teacher_outputs = self.teacher(**inputs)
            teacher_logits = teacher_outputs.logits

        # Calculate the distillation loss using KL Divergence
        loss = F.kl_div(
            F.log_softmax(student_logits / self.temperature, dim=-1),
            F.softmax(teacher_logits / self.temperature, dim=-1),
            reduction="batchmean"
        ) * (self.temperature ** 2)

        # Adding cross-entropy loss
        loss += F.cross_entropy(student_logits, inputs["labels"])

        return (loss, student_outputs) if return_outputs else loss

# Training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
)

# Initialize the distillation trainer
trainer = DistillationTrainer(
    teacher_model=teacher_model,
    model=student_model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation_matched'],
    tokenizer=student_tokenizer,
    temperature=2.0  # Higher temperature smooths the probabilities from the teacher
)

# Train the student model
trainer.train()

# Evaluate the student model on the validation set
trainer.evaluate()

# Example for prediction
def predict_sentiment(text_premise, text_hypothesis):
    inputs = student_tokenizer(text_premise, text_hypothesis, return_tensors="pt", truncation=True, padding=True)
    outputs = student_model(**inputs)
    predicted_class = torch.argmax(outputs.logits, dim=-1).item()
    label_map = {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
    return label_map[predicted_class]

# Example usage
text_premise = "The dog is flying in the park."
text_hypothesis = "The dog is beyond death."
predicted_sentiment = predict_sentiment(text_premise, text_hypothesis)
print(f"Predicted sentiment: {predicted_sentiment}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 6.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 14.0.2
    Uninstalling pyarrow-14.0.2:
      Successfully uninstalled pyarrow-14.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 requires pyarrow<15.0.0a0,>=14.0.1, but you have pyarrow 17.0.0 which is incompatible.
ibis-framework 8.0.0 requires pyarrow<16,>=2, but you have pyarrow 17.0.0 which is incompatible.


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating validation_matched split:   0%|          | 0/9815 [00:00<?, ? examples/s]

Generating validation_mismatched split:   0%|          | 0/9832 [00:00<?, ? examples/s]

...

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


ValueError: --load_best_model_at_end requires the save and eval strategy to match, but found
- Evaluation strategy: epoch
- Save strategy: steps

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoConfig, Trainer, TrainingArguments)

# Load the data
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
dataset2 = pd.read_parquet("/content/drive/MyDrive/default_train_0000.parquet", engine="pyarrow")
dataset3 = pd.read_parquet("/content/drive/MyDrive/default_validation_0000.parquet", engine="pyarrow")

# Tokenize the data
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tdataset2_text = tokenizer(dataset2["text"].tolist(), padding=True, truncation=True, max_length=128)
tdataset3_text = tokenizer(dataset3["text"].tolist(), padding=True, truncation=True, max_length=128)

# Create PyTorch tensors
labels2 = torch.tensor(dataset2['label'].values)
labels3 = torch.tensor(dataset3['label'].values)

input_ids2 = torch.tensor(tdataset2_text['input_ids'])
token_type_ids2 = torch.tensor(tdataset2_text['token_type_ids'])
attention_mask2 = torch.tensor(tdataset2_text['attention_mask'])

input_ids3 = torch.tensor(tdataset3_text['input_ids'])
token_type_ids3 = torch.tensor(tdataset3_text['token_type_ids'])
attention_mask3 = torch.tensor(tdataset3_text['attention_mask'])

from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self, input_ids, token_type_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.token_type_ids = token_type_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'token_type_ids': self.token_type_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }

# Create dataset instances
tensor_dataset2 = MyDataset(input_ids2, token_type_ids2, attention_mask2, labels2)
tensor_dataset3 = MyDataset(input_ids3, token_type_ids3, attention_mask3, labels3)


# Create PyTorch datasets
#tensor_dataset2 = TensorDataset(input_ids2, token_type_ids2, attention_mask2, labels2)
#tensor_dataset3 = TensorDataset(input_ids3, token_type_ids3, attention_mask3, labels3)

# Create DataLoaders
BATCH_SIZE = 8
data_loader2 = DataLoader(tensor_dataset2, batch_size=BATCH_SIZE, shuffle=True)
data_loader3 = DataLoader(tensor_dataset3, batch_size=BATCH_SIZE, shuffle=False)


# Define the custom Trainer class
class ImageDistilTrainer(Trainer):
    def __init__(self, student_model=None, teacher_model=None, temperature=1.0, *args, **kwargs):
        super().__init__(model=student_model, *args, **kwargs)
        self.teacher = teacher_model
        self.temperature = temperature

    def compute_loss(self, model, inputs, return_outputs=False):
        student_outputs = model(**inputs)
        with torch.no_grad():
            teacher_outputs = self.teacher(**inputs)

        # Calculate distillation loss using logits or predictions
        student_logits = student_outputs.logits / self.temperature
        teacher_logits = teacher_outputs.logits / self.temperature

        # Use predictions for distillation loss if num_labels is 1
        if student_outputs.logits.shape[1] == 1:
            loss = F.kl_div(
                F.logsigmoid(student_logits.squeeze()),
                torch.sigmoid(teacher_logits.squeeze()),
                reduction="batchmean"
            )
            # For binary classification, use BCEWithLogitsLoss
            loss += F.binary_cross_entropy_with_logits(
                student_logits.squeeze(),
                inputs['labels'].float()
            )
        else:
            loss = F.kl_div(F.log_softmax(student_logits, dim=-1),
                            F.softmax(teacher_logits, dim=-1),
                            reduction="batchmean")
            # For multi-class classification, use cross_entropy
            loss += F.cross_entropy(student_logits, inputs['labels'])

        return (loss, student_outputs) if return_outputs else loss

# Load the models
config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=1)
student_model = AutoModelForSequenceClassification.from_config(config)
teacher_model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', config=config)

# Freeze teacher model
for param in teacher_model.parameters():
    param.requires_grad = False


# Define training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/output",
    num_train_epochs=3,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    logging_dir="/content/drive/MyDrive/logging",
    logging_strategy="epoch",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    fp16=True,
)


print(type(tensor_dataset2[1]))
print(tensor_dataset2[1])
trainer = ImageDistilTrainer(
    student_model=student_model,
    teacher_model=teacher_model,
    train_dataset=tensor_dataset2,
    eval_dataset=tensor_dataset3,
    args=training_args,
    temperature=10.0,
)

trainer.train()

ValueError: mount failed

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoConfig, Trainer, TrainingArguments)

# Load the data
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
dataset2 = pd.read_parquet("/content/drive/MyDrive/default_train_0000.parquet", engine="pyarrow")
dataset3 = pd.read_parquet("/content/drive/MyDrive/default_validation_0000.parquet", engine="pyarrow")

# Tokenize the data
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tdataset2_text = tokenizer(dataset2["text"].tolist(), padding=True, truncation=True, max_length=128)
tdataset3_text = tokenizer(dataset3["text"].tolist(), padding=True, truncation=True, max_length=128)

# Create PyTorch tensors
labels2 = torch.tensor(dataset2['label'].values)
labels3 = torch.tensor(dataset3['label'].values)

input_ids2 = torch.tensor(tdataset2_text['input_ids'])
token_type_ids2 = torch.tensor(tdataset2_text['token_type_ids'])
attention_mask2 = torch.tensor(tdataset2_text['attention_mask'])

input_ids3 = torch.tensor(tdataset3_text['input_ids'])
token_type_ids3 = torch.tensor(tdataset3_text['token_type_ids'])
attention_mask3 = torch.tensor(tdataset3_text['attention_mask'])

from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self, input_ids, token_type_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.token_type_ids = token_type_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'token_type_ids': self.token_type_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }

# Create dataset instances
tensor_dataset2 = MyDataset(input_ids2, token_type_ids2, attention_mask2, labels2)
tensor_dataset3 = MyDataset(input_ids3, token_type_ids3, attention_mask3, labels3)


# Create PyTorch datasets
#tensor_dataset2 = TensorDataset(input_ids2, token_type_ids2, attention_mask2, labels2)
#tensor_dataset3 = TensorDataset(input_ids3, token_type_ids3, attention_mask3, labels3)

# Create DataLoaders
BATCH_SIZE = 8
data_loader2 = DataLoader(tensor_dataset2, batch_size=BATCH_SIZE, shuffle=True)
data_loader3 = DataLoader(tensor_dataset3, batch_size=BATCH_SIZE, shuffle=False)


# Define the custom Trainer class
class ImageDistilTrainer(Trainer):
    def __init__(self, student_model=None, teacher_model=None, temperature=1.0, *args, **kwargs):
        super().__init__(model=student_model, *args, **kwargs)
        self.teacher = teacher_model
        self.temperature = temperature

    def compute_loss(self, model, inputs, return_outputs=False):
        student_outputs = model(**inputs)
        with torch.no_grad():
            teacher_outputs = self.teacher(**inputs)

        # Calculate distillation loss using logits or predictions
        student_logits = student_outputs.logits / self.temperature
        teacher_logits = teacher_outputs.logits / self.temperature

        # Use predictions for distillation loss if num_labels is 1
        if student_outputs.logits.shape[1] == 1:
            loss = F.kl_div(
                F.logsigmoid(student_logits.squeeze()),
                torch.sigmoid(teacher_logits.squeeze()),
                reduction="batchmean"
            )
            # For binary classification, use BCEWithLogitsLoss
            loss += F.binary_cross_entropy_with_logits(
                student_logits.squeeze(),
                inputs['labels'].float()
            )
        else:
            loss = F.kl_div(F.log_softmax(student_logits, dim=-1),
                            F.softmax(teacher_logits, dim=-1),
                            reduction="batchmean")
            # For multi-class classification, use cross_entropy
            loss += F.cross_entropy(student_logits, inputs['labels'])

        return (loss, student_outputs) if return_outputs else loss

# Load the models
config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=1)
student_model = AutoModelForSequenceClassification.from_config(config)
teacher_model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', config=config)

# Freeze teacher model
for param in teacher_model.parameters():
    param.requires_grad = False


# Define training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/output",
    num_train_epochs=3,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    logging_dir="/content/drive/MyDrive/logging",
    logging_strategy="epoch",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    fp16=True,
)


print(type(tensor_dataset2[1]))
print(tensor_dataset2[1])
trainer = ImageDistilTrainer(
    student_model=student_model,
    teacher_model=teacher_model,
    train_dataset=tensor_dataset2,
    eval_dataset=tensor_dataset3,
    args=training_args,
    temperature=10.0,
)

trainer.train()

Mounted at /content/drive


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1494: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


<class 'dict'>
{'input_ids': tensor([  101,  1002, 10507,  2140,  1002, 22110,  2140,  1011,  2053, 16069,
         2685,  2000, 21725,  2015, 11251,  2012, 11485,  1998,  2548,  7139,
        16770,  1024,  1013,  1013,  1056,  1012,  2522,  1013,  1061,  2290,
         3501, 13876,  2475,  5596,  2509,   102,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0]), 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 'attention_mask': tensor(

Epoch,Training Loss,Validation Loss
1,-1.732600,-2.728952


In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoConfig, Trainer, TrainingArguments)

# Load the data
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
dataset2 = pd.read_parquet("/content/drive/MyDrive/default_train_0000.parquet", engine="pyarrow")
dataset3 = pd.read_parquet("/content/drive/MyDrive/default_validation_0000.parquet", engine="pyarrow")

# Tokenize the data
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tdataset2_text = tokenizer(dataset2["text"].tolist(), padding=True, truncation=True, max_length=128)
tdataset3_text = tokenizer(dataset3["text"].tolist(), padding=True, truncation=True, max_length=128)

# Create PyTorch tensors
labels2 = torch.tensor(dataset2['label'].values)
labels3 = torch.tensor(dataset3['label'].values)

input_ids2 = torch.tensor(tdataset2_text['input_ids'])
token_type_ids2 = torch.tensor(tdataset2_text['token_type_ids'])
attention_mask2 = torch.tensor(tdataset2_text['attention_mask'])

input_ids3 = torch.tensor(tdataset3_text['input_ids'])
token_type_ids3 = torch.tensor(tdataset3_text['token_type_ids'])
attention_mask3 = torch.tensor(tdataset3_text['attention_mask'])

from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self, input_ids, token_type_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.token_type_ids = token_type_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'token_type_ids': self.token_type_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'labels': self.labels[idx]
        }

# Create dataset instances
tensor_dataset2 = MyDataset(input_ids2, token_type_ids2, attention_mask2, labels2)
tensor_dataset3 = MyDataset(input_ids3, token_type_ids3, attention_mask3, labels3)


# Create PyTorch datasets
#tensor_dataset2 = TensorDataset(input_ids2, token_type_ids2, attention_mask2, labels2)
#tensor_dataset3 = TensorDataset(input_ids3, token_type_ids3, attention_mask3, labels3)

# Create DataLoaders
BATCH_SIZE = 8
data_loader2 = DataLoader(tensor_dataset2, batch_size=BATCH_SIZE, shuffle=True)
data_loader3 = DataLoader(tensor_dataset3, batch_size=BATCH_SIZE, shuffle=False)


# Define the custom Trainer class
class ImageDistilTrainer(Trainer):
    def __init__(self, student_model=None, teacher_model=None, temperature=1.0, *args, **kwargs):
        super().__init__(model=student_model, *args, **kwargs)
        self.teacher = teacher_model
        self.temperature = temperature

    def compute_loss(self, model, inputs, return_outputs=False):
        student_outputs = model(**inputs)
        with torch.no_grad():
            teacher_outputs = self.teacher(**inputs)

        # Calculate distillation loss using logits or predictions
        student_logits = student_outputs.logits / self.temperature
        teacher_logits = teacher_outputs.logits / self.temperature

        # Use predictions for distillation loss if num_labels is 1
        if student_outputs.logits.shape[1] == 1:
            loss = F.kl_div(
                F.logsigmoid(student_logits.squeeze()),
                torch.sigmoid(teacher_logits.squeeze()),
                reduction="batchmean"
            )
            # For binary classification, use BCEWithLogitsLoss
            loss += F.binary_cross_entropy_with_logits(
                student_logits.squeeze(),
                inputs['labels'].float()
            )
        else:
            loss = F.kl_div(F.log_softmax(student_logits, dim=-1),
                            F.softmax(teacher_logits, dim=-1),
                            reduction="batchmean")
            # For multi-class classification, use cross_entropy
            loss += F.cross_entropy(student_logits, inputs['labels'])

        return (loss, student_outputs) if return_outputs else loss

# Load the models
config = AutoConfig.from_pretrained('bert-base-uncased', num_labels=1)
student_model = AutoModelForSequenceClassification.from_config(config)
teacher_model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', config=config)

# Freeze teacher model
for param in teacher_model.parameters():
    param.requires_grad = False


# Define training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/output",
    num_train_epochs=3,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    logging_dir="/content/drive/MyDrive/logging",
    logging_strategy="epoch",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    fp16=True,
)


print(type(tensor_dataset2[1]))
print(tensor_dataset2[1])
trainer = ImageDistilTrainer(
    student_model=student_model,
    teacher_model=teacher_model,
    train_dataset=tensor_dataset2,
    eval_dataset=tensor_dataset3,
    args=training_args,
    temperature=10.0,
)

trainer.train()

Mounted at /content/drive


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1494: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


<class 'dict'>
{'input_ids': tensor([  101,  1002, 10507,  2140,  1002, 22110,  2140,  1011,  2053, 16069,
         2685,  2000, 21725,  2015, 11251,  2012, 11485,  1998,  2548,  7139,
        16770,  1024,  1013,  1013,  1056,  1012,  2522,  1013,  1061,  2290,
         3501, 13876,  2475,  5596,  2509,   102,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0]), 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0]), 'attention_mask': tensor(

Epoch,Training Loss,Validation Loss
1,-1.787800,-2.777681
2,-3.284300,-3.837504
3,-3.971300,-4.192932


TrainOutput(global_step=3579, training_loss=-3.0144311052755657, metrics={'train_runtime': 38334.1996, 'train_samples_per_second': 0.747, 'train_steps_per_second': 0.093, 'total_flos': 1206383162649948.0, 'train_loss': -3.0144311052755657, 'epoch': 3.0})

In [ ]:
import torch
import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datasets import load_dataset, load_metric
import numpy as np
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)

# Define model IDs
teacher_id = "roberta-large"
student_id = "distilbert-base-uncased"

# Load tokenizers
teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_id)
student_tokenizer = AutoTokenizer.from_pretrained(student_id)

# Load the MNLI dataset using datasets library
logging.info("Loading dataset...")
dataset = load_dataset("glue", "mnli")

# Tokenize the dataset
def process(examples):
    return teacher_tokenizer(examples['premise'], examples['hypothesis'], truncation=True, padding=True, max_length=512)

tokenized_datasets = dataset.map(process, batched=True)

# Rename label column to "labels" (required by Hugging Face Trainer)
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

# Load teacher and student models
logging.info("Loading teacher and student models...")
teacher_model = AutoModelForSequenceClassification.from_pretrained(teacher_id, num_labels=3)
student_model = AutoModelForSequenceClassification.from_pretrained(student_id, num_labels=3)

# Freeze the teacher model (no training)
for param in teacher_model.parameters():
    param.requires_grad = False

# Define custom distillation loss with KL Divergence and Cross Entropy
class DistillationTrainer(Trainer):
    def __init__(self, *args, teacher_model=None, temperature=2.0, alpha=0.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher = teacher_model
        self.temperature = temperature
        self.alpha = alpha
        self.teacher.eval()

    def compute_loss(self, model, inputs, return_outputs=False):
        # Get student model outputs
        outputs_student = model(**inputs)
        student_loss = outputs_student.loss

        # Get teacher model outputs (with no gradient calculation)
        with torch.no_grad():
            outputs_teacher = self.teacher(**inputs)

        # Ensure the shapes of student and teacher logits match
        assert outputs_student.logits.size() == outputs_teacher.logits.size()

        # Compute distillation loss (KLDivLoss)
        loss_function = torch.nn.KLDivLoss(reduction="batchmean")
        distillation_loss = loss_function(
            F.log_softmax(outputs_student.logits / self.temperature, dim=-1),
            F.softmax(outputs_teacher.logits / self.temperature, dim=-1)
        ) * (self.temperature ** 2)

        # Final loss is a weighted sum of student loss and distillation loss
        loss = self.alpha * student_loss + (1 - self.alpha) * distillation_loss

        return (loss, outputs_student) if return_outputs else loss

# Training arguments
training_args = TrainingArguments(
    output_dir="./distilbert-mnli-distilled",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    save_total_limit=1
)

# Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=teacher_tokenizer)

# Load accuracy metric
metric = load_metric("loss")

# Define the compute metrics function for evaluation
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

# Initialize the distillation trainer
trainer = DistillationTrainer(
    model=student_model,
    args=training_args,
    teacher_model=teacher_model,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation_matched"],
    data_collator=data_collator,
    tokenizer=student_tokenizer,
    compute_metrics=compute_metrics,
    temperature=2.0,  # Temperature for distillation
    alpha=0.5  # Balance between student loss and distillation loss
)

# Train the model
logging.info("Training the student model with distillation...")
trainer.train()

# Evaluate the student model
logging.info("Evaluating the student model...")
metrics = trainer.evaluate()
logging.info(f"Evaluation metrics: {metrics}")

# Save the trained student model
logging.info("Saving the trained student model...")
trainer.save_model()


Map:   0%|          | 0/392702 [00:00<?, ? examples/s]

Map:   0%|          | 0/9815 [00:00<?, ? examples/s]

Map:   0%|          | 0/9832 [00:00<?, ? examples/s]

Map:   0%|          | 0/9796 [00:00<?, ? examples/s]

Map:   0%|          | 0/9847 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-3-a55b871949e2>:101: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  metric = load_metric("loss")


FileNotFoundError: Couldn't find a metric script at /content/loss/loss.py. Metric 'loss' doesn't exist on the Hugging Face Hub either.

In [ ]:
eval_results = trainer.evaluate(tf_val_dataset)
print(f"Test accuracy: {eval_results['eval_accuracy']:.4f}")

input_text = input('Enter a sentence: ')
input_ids = tokenizer.encode_plus(input_text,
                                   add_special_tokens=True,
                                   max_length=max_length,
                                   return_attention_mask=True,
                                   return_tensors='tf',
                                   padding='max_length',
                                   truncation=True)

output = student_model(input_ids['input_ids'], attention_mask=input_ids['attention_mask'])
print(output.logits)
scores = output[0][0].detach().numpy()
scores = softmax(scores)
#ranking = np.argsort(scores)
#ranking = ranking[::-1]
labels={ 0: "positive", 1: "neutral", 2: "negative"}
ranking = np.argsort(scores)
ranking = ranking[::-1]
for i in range(scores.shape[0]):
    l = labels[ranking[i]]
    s = scores[ranking[i]]
    print(f"{i + 1}) {l} {np.round(float(s), 4)}")